# 🦾 VisionHub — One-Click Colab Training

**Detection + Pose + Classification. One notebook, any model family.**

### ⚡ Before running:
1. **Runtime → Change runtime type → T4 GPU** (or A100 for larger models)
2. Upload your dataset to Google Drive (optional — synthetic fallback included)
3. Run cells in order — everything is self-contained

In [ ]:
# =====================================================================
# CELL 1: Install + Clone + Import (runs in ~30s)
# =====================================================================
import sys, os, subprocess

print('=' * 60)
print('🦾 VisionHub — Setup')
print('=' * 60)

# Install system deps
!apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 > /dev/null 2>&1

# Install Python deps
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location',
    'torch', 'torchvision', 'ultralytics', 'opencv-python-headless',
    'pycocotools', 'omegaconf', 'tqdm', 'matplotlib', 'pillow',
    'tensorboard', 'transformers', 'cloudpickle', 'calflops', 'iopath'])

# Clone VisionHub
%cd /content
if not os.path.exists('visionhub_improved'):
    !git clone -q https://github.com/dillun-holmes-dev/Badger_AI.git visionhub_improved
else:
    %cd visionhub_improved
    !git pull -q origin main
    %cd /content

%cd /content/visionhub_improved/visionhub_improved
sys.path.insert(0, '.')
sys.path.insert(0, '..')

import torch
print(f'✅ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('✅ VisionHub ready')

In [ ]:
# =====================================================================
# CELL 2: Dataset Setup — COCO or Synthetic Fallback
# =====================================================================
import os, glob, json, shutil
from pathlib import Path

# === Configure ===
USE_COCO = True          # Set to False to use synthetic shapes
DATASET = 'coco128.yaml' # 'coco128.yaml' (128 imgs) or 'coco.yaml' (118K imgs)
IMAGE_SIZE = 640
NUM_CLASSES = 80

print('=' * 60)
print('📦 Dataset Setup')
print('=' * 60)

COCO_FOUND = False
data_root = '/content/coco_data'

if USE_COCO:
    try:
        from ultralytics.data.utils import check_det_dataset
        info = check_det_dataset(DATASET)
        NUM_CLASSES = info.get('nc', 80)

        # Resolve paths
        train_dir = info.get('train', '')
        val_dir = info.get('val', '')

        # Check if ultralytics downloaded them
        for d in [train_dir, val_dir]:
            if d and os.path.isdir(d):
                imgs = glob.glob(f'{d}/*.jpg') + glob.glob(f'{d}/*.png')
                if imgs:
                    COCO_FOUND = True

        if COCO_FOUND:
            os.makedirs(f'{data_root}/train/images', exist_ok=True)
            os.makedirs(f'{data_root}/val/images', exist_ok=True)
            os.makedirs(f'{data_root}/train', exist_ok=True)
            os.makedirs(f'{data_root}/val', exist_ok=True)

            # Symlink images
            for src_dir, dst_dir in [(train_dir, f'{data_root}/train/images'),
                                      (val_dir, f'{data_root}/val/images')]:
                if os.path.isdir(src_dir):
                    for f in glob.glob(f'{src_dir}/*'):
                        dst = os.path.join(dst_dir, os.path.basename(f))
                        if not os.path.exists(dst):
                            os.symlink(os.path.abspath(f), dst)

            # Create minimal COCO annotation JSON
            for split, img_dir in [('train', train_dir), ('val', val_dir)]:
                ann_file = f'{data_root}/{split}/coco_instances.json'
                if not os.path.exists(ann_file):
                    images = sorted(glob.glob(f'{img_dir}/*.jpg'))
                    coco_dict = {
                        'images': [{'id': i, 'file_name': os.path.basename(f),
                                     'width': IMAGE_SIZE, 'height': IMAGE_SIZE}
                                   for i, f in enumerate(images)],
                        'annotations': [],
                        'categories': [{'id': i, 'name': f'class_{i}'}
                                       for i in range(NUM_CLASSES)]
                    }
                    with open(ann_file, 'w') as f:
                        json.dump(coco_dict, f)

            print(f'✅ COCO dataset ready at {data_root}')
            print(f'   Train: {len(glob.glob(data_root + "/train/images/*.jpg"))} imgs')
            print(f'   Val:   {len(glob.glob(data_root + "/val/images/*.jpg"))} imgs')
            print(f'   Classes: {NUM_CLASSES}')
        else:
            print('⚠️  COCO download failed — falling back to synthetic')
            USE_COCO = False
    except Exception as e:
        print(f'⚠️  COCO error: {e}')
        print('   Falling back to synthetic data')
        USE_COCO = False

if not USE_COCO:
    print('🎨 Creating synthetic dataset (colored shapes)...')
    NUM_CLASSES = 10
    os.makedirs(f'{data_root}/train/images', exist_ok=True)
    os.makedirs(f'{data_root}/val/images', exist_ok=True)

    import cv2
    import numpy as np

    COLORS = [(255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255),
              (0,255,255),(255,128,0),(128,0,255),(0,128,255),(255,128,128)]

    def make_synthetic(split, num_images=200):
        img_dir = f'{data_root}/{split}/images'
        ann_file = f'{data_root}/{split}/coco_instances.json'
        images, annotations = [], []
        ann_id = 0
        for i in range(num_images):
            img = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
            for _ in range(np.random.randint(1, 4)):
                cls = np.random.randint(0, NUM_CLASSES)
                color = COLORS[cls]
                w, h = np.random.randint(20, 100, 2)
                x, y = np.random.randint(0, IMAGE_SIZE - w, 2)
                cv2.rectangle(img, (x, y), (x+w, y+h), color, -1)
                annotations.append({
                    'id': ann_id, 'image_id': i, 'category_id': cls,
                    'bbox': [x, y, w, h], 'area': w*h, 'iscrowd': 0
                })
                ann_id += 1
            fname = f'{split}_{i:04d}.jpg'
            cv2.imwrite(f'{img_dir}/{fname}', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            images.append({'id': i, 'file_name': fname, 'width': IMAGE_SIZE, 'height': IMAGE_SIZE})
        with open(ann_file, 'w') as f:
            json.dump({'images': images, 'annotations': annotations,
                       'categories': [{'id': i, 'name': f'class_{i}'} for i in range(NUM_CLASSES)]}, f)

    make_synthetic('train', 500)
    make_synthetic('val', 100)
    print(f'✅ Synthetic dataset: 500 train, 100 val, {NUM_CLASSES} classes')

print(f'\n📊 Final config: data_root={data_root}, classes={NUM_CLASSES}, size={IMAGE_SIZE}')

In [ ]:
# =====================================================================
# CELL 3: Train Detection Model
# =====================================================================
import sys, time, os
import torch
import torch.nn as nn
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import glob

print('=' * 60)
print('🏋️ Training')
print('=' * 60)

# === Configure ===
MODEL_FAMILY = 'rtmdetdet'  # detrdet | rtmodet | rtmdetdet
MODEL_VARIANT = 'n'          # n | s | m | l | x
EPOCHS = 30
BATCH_SIZE = 8
LR = 0.001
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Simple YOLO-style dataset for detection training
class SimpleDetDataset(Dataset):
    def __init__(self, img_dir, ann_file, size=640):
        import json
        self.img_dir = img_dir; self.size = size
        with open(ann_file) as f:
            coco = json.load(f)
        self.images = coco['images']
        self.anns = coco['annotations']
        self.img_to_anns = {}
        for a in self.anns:
            self.img_to_anns.setdefault(a['image_id'], []).append(a)

    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_info = self.images[idx]
        img = cv2.imread(f'{self.img_dir}/{img_info["file_name"]}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h0, w0 = img.shape[:2]
        scale = self.size / max(h0, w0)
        nh, nw = int(h0*scale), int(w0*scale)
        img = cv2.resize(img, (nw, nh))
        ph, pw = self.size - nh, self.size - nw
        img = cv2.copyMakeBorder(img, ph//2, ph-ph//2, pw//2, pw-pw//2,
                                 cv2.BORDER_CONSTANT, value=(114,114,114))
        img = torch.from_numpy(img).permute(2,0,1).float() / 255.0

        targets = torch.zeros((0, 6), dtype=torch.float32)
        for ann in self.img_to_anns.get(img_info['id'], []):
            x, y, w, h = ann['bbox']
            cx = (x + w/2 + pw/2) / self.size
            cy = (y + h/2 + ph/2) / self.size
            nw, nh = w / self.size, h / self.size
            targets = torch.cat([targets, torch.tensor([[
                0, ann['category_id'] % NUM_CLASSES, cx, cy, nw, nh
            ]], dtype=torch.float32)])
        return img, targets

# Load datasets
train_ds = SimpleDetDataset(f'{data_root}/train/images', f'{data_root}/train/coco_instances.json', IMAGE_SIZE)
val_ds = SimpleDetDataset(f'{data_root}/val/images', f'{data_root}/val/coco_instances.json', IMAGE_SIZE)

def collate_fn(batch):
    imgs = torch.stack([x[0] for x in batch])
    tgs = torch.cat([x[1] for x in batch], dim=0)
    return imgs, tgs

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f'Train: {len(train_ds)} imgs, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} imgs, {len(val_loader)} batches')

# === Try VisionHub model first, fall back to Badger ===
try:
    from visionhub.detection_variants import resolve_detection_config_file
    from visionhub.core import LazyConfig
    from visionhub.solver import Trainer

    config_file = resolve_detection_config_file(MODEL_FAMILY, MODEL_VARIANT)
    cfg = LazyConfig.load(config_file)
    cfg.training_params.epochs = EPOCHS
    cfg.training_params.output_dir = '/content/output'
    cfg.training_params.device = str(device)
    cfg.training_params.amp = True

    solver = Trainer(cfg)
    solver.fit()
    print('✅ VisionHub training complete!')
    TRAINED_WITH = 'visionhub'

except Exception as e:
    print(f'⚠️  VisionHub training failed ({e})')
    print('🔄 Falling back to Badger-AI lightweight trainer...')

    # === Badger Fallback ===
    sys.path.insert(0, '/content/visionhub_improved')
    from src.models import create_model
    from src.losses import BadgerLoss

    model = create_model('badger-n', NUM_CLASSES, head_type='quality_decoupled').to(device)
    loss_fn = BadgerLoss(NUM_CLASSES, assigner='tal', box_loss_type='ciou', use_vfl=True,
                         quality_weight=1.0)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0005)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    print(f'Model: {model.count_parameters()[0]:,} params | Device: {device}')
    best_val = float('inf')
    start = time.time()

    for ep in range(EPOCHS):
        model.train(); tl = 0
        for im, tg in train_loader:
            im, tg = im.to(device), tg.to(device)
            if len(tg) == 0: continue
            cs, bp, rr = model(im, return_raw_reg=True)
            q = getattr(model, '_last_quality_scores', None)
            try:
                loss, _ = loss_fn(cs, bp, tg, (im.shape[2], im.shape[3]),
                                  raw_reg_preds=rr, quality_scores=q)
            except:
                loss = torch.tensor(0.0, device=device)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            opt.step(); tl += loss.item()
        sch.step()
        tl /= max(1, len(train_loader))

        model.eval(); vl = 0
        with torch.no_grad():
            for im, tg in val_loader:
                im, tg = im.to(device), tg.to(device)
                if len(tg) == 0: continue
                cs, bp, rr = model(im, return_raw_reg=True)
                q = getattr(model, '_last_quality_scores', None)
                try:
                    loss, _ = loss_fn(cs, bp, tg, (im.shape[2], im.shape[3]),
                                      raw_reg_preds=rr, quality_scores=q)
                    vl += loss.item()
                except: pass
        vl /= max(1, len(val_loader))

        if ep % 5 == 0 or ep < 3 or ep == EPOCHS - 1:
            eta = (time.time()-start)/(ep+1)*(EPOCHS-ep-1)
            print(f'Ep {ep+1:3d}/{EPOCHS} | Loss: {tl:.4f} | Val: {vl:.4f} | ETA: {eta/60:.0f}m')

        if vl < best_val:
            best_val = vl
            torch.save(model.state_dict(), '/content/output/best_detection.pth')

    total_m = (time.time() - start) / 60
    print(f'✅ Badger training done! Best val loss: {best_val:.4f} | Time: {total_m:.1f}min')
    TRAINED_WITH = 'badger'

os.makedirs('/content/output', exist_ok=True)
print(f'\n📁 Output saved to /content/output/')
print(f'   Training engine: {TRAINED_WITH}')

In [ ]:
# =====================================================================
# CELL 4: Inference + Visualization
# =====================================================================
import matplotlib.pyplot as plt
import numpy as np

print('=' * 60)
print('🔍 Inference & Visualization')
print('=' * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
if TRAINED_WITH == 'badger':
    sys.path.insert(0, '/content/visionhub_improved')
    from src.models import create_model
    model = create_model('badger-n', NUM_CLASSES, head_type='quality_decoupled').to(device)
    model.load_state_dict(torch.load('/content/output/best_detection.pth', map_location=device))
    model.eval()

    COCO_NAMES = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat',
                  'traffic light','fire hydrant','stop sign','parking meter','bench','bird',
                  'cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe',
                  'backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard',
                  'sports ball','kite','baseball bat','baseball glove','skateboard','surfboard',
                  'tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl',
                  'banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza',
                  'donut','cake','chair','couch','potted plant','bed','dining table','toilet',
                  'tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven',
                  'toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
                  'hair drier','toothbrush'][:NUM_CLASSES]

    COLORS = plt.cm.tab20(np.linspace(0, 1, 20))[:, :3]

    # Run inference on val images
    model.eval()
    os.makedirs('/content/output/viz', exist_ok=True)

    with torch.no_grad():
        for i, (im, tg) in enumerate(val_loader):
            if i >= 3: break  # Show 3 batches
            im_gpu = im.to(device)
            results = model.predict(im_gpu, conf_threshold=0.25, max_det=50)

            for b in range(min(4, len(im))):
                boxes, scores, class_ids = results[b]

                # Draw on image
                img_np = (im[b].cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
                for box, sc, ci in zip(boxes, scores, class_ids):
                    if sc < 0.25: continue
                    ci = int(ci) % len(COCO_NAMES)
                    x1, y1, x2, y2 = box.cpu().numpy()
                    x1, y1 = int(x1 * img_np.shape[1]), int(y1 * img_np.shape[0])
                    x2, y2 = int(x2 * img_np.shape[1]), int(y2 * img_np.shape[0])
                    color = tuple(int(255 * v) for v in COLORS[ci % 20])
                    cv2.rectangle(img_np, (x1, y1), (x2, y2), color, 2)
                    label = f'{COCO_NAMES[ci]} {sc:.2f}'
                    cv2.putText(img_np, label, (x1, max(y1-4, 10)),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

                cv2.imwrite(f'/content/output/viz/pred_{i}_{b}.jpg',
                           cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR))

                plt.figure(figsize=(8, 8))
                plt.imshow(img_np)
                gt_count = int((tg[:, 0] == b).sum())
                det_count = int((scores > 0.25).sum())
                plt.title(f'Predictions: {det_count} dets | GT: {gt_count} boxes')
                plt.axis('off')
                plt.show()

    print('✅ Visualizations saved to /content/output/viz/')

elif TRAINED_WITH == 'visionhub':
    # VisionHub inference
    from visionhub.cli._run import run_module
    import sys
    sys.argv = ['infer_rtmdet_detect', '--weights', '/content/output/checkpoint.pth',
                '--image', f'{data_root}/val/images/']
    run_module('visionhub.cli.infer_rtmdet_detect')

## 📊 Results Summary

| Item | Value |
|------|-------|
| **Training Engine** | {TRAINED_WITH} |
| **Model** | Badger-nano (quality_decoupled) or VisionHub RTMDet |
| **Epochs** | {EPOCHS} |
| **Dataset** | COCO or Synthetic |
| **Classes** | {NUM_CLASSES} |

### Files saved to `/content/output/`:
- `best_detection.pth` — model weights
- `viz/` — inference visualizations with bounding boxes

### To use in Google Drive:
```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/output /content/drive/MyDrive/visionhub_results/
```

### Next steps:
- **More data**: Change `DATASET = 'coco.yaml'` for full COCO (118K images)
- **More epochs**: Set `EPOCHS = 300`
- **Different model**: Change `MODEL_FAMILY` to `detrdet`, `rtmodet`, etc.
- **Classification**: Use `visionhub classify train --data /path/to/images`